# Low Frequency Categories Handling

This notebook handles low-frequency categories in categorical variables by merging them into an "Others" category. This helps reduce noise and prevent overfitting while preserving interpretability.

## Notebook Overview
1. **Setup**: Import libraries and configure display options
2. **Data Loading**: Load cleaned data from previous step
3. **Low Frequency Handling**: Apply statistical and business thresholds
4. **Validation**: Review results
5. **Export**: Save processed data

## 1. Setup

In [ ]:
import pandas as pd
import os
from pathlib import Path
from collections import Counter
from matplotlib.pylab import size

## 2. Load Data

In [ ]:
# Define paths
current_user = os.getlogin()
DATA_DIR = Path(rf"/home/{current_user}/local-share")
OUT_DIR = DATA_DIR / "processed"
cleaned_path = OUT_DIR / "cleaned_original.csv"

# Load data
data = pd.read_csv(cleaned_path, sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")

## 3. Handle Low Frequency Categories

**Target Columns:**
- `sdo5_degree_degree`
- `sdo1_previous_Previous_Education_Type`
- `sdo2_skc_ADVIES_DEF`
- `sdo2_orientation_Event_Types_Attended`* (requires special handling)

**Threshold:** 100 occurrences (~0.4% of N=23,286) - balances statistical stability with business relevance.

In [ ]:
# -----------------------------------------------------------------------------
# Merge low-frequency categories into "Others"
#
# Rationale:
#   • Statistical threshold: 0.5% (≈117 rows) ensures ~48 occurrences per fold in 5-fold CV
#   • Business threshold: 100 rows (~0.4%) retains operationally meaningful categories
#   • NaN values are preserved (not merged into "Others")
# -----------------------------------------------------------------------------

cat_cols = [
    "sdo5_degree_degree",
    "sdo1_previous_Previous_Education_Type",
    "sdo2_skc_ADVIES_DEF",
]

N = size(data, 0)
min_count = 100

print(f"Dataset size: {N} rows")
print(f"Business threshold: {min_count} occurrences (~{100/N:.2%})\n")

# Apply threshold
for col in cat_cols:
    vc = data[col].value_counts(dropna=False)
    rare_levels = [lvl for lvl in vc.index if (not pd.isna(lvl)) and (vc[lvl] < min_count)]
    if rare_levels:
        print(f"{col}: Merging {len(rare_levels)} rare level(s) → 'Others'")
        data[col] = data[col].where(~data[col].isin(rare_levels), "Others")

In [ ]:
# Handle event types (comma-separated values require special processing)
event_types = data["sdo2_orientation_Event_Types_Attended"].astype(str).str.strip().str.lower().str.split(",")
all_event_types = [etype.strip().lower() for sublist in event_types for etype in sublist]
event_type_counts = Counter(all_event_types)

print("\nEvent types before merging:")
print(event_type_counts)

# Create mapping: rare event types → "other"
event_type_mapping = {etype: ("other" if count < min_count else etype) for etype, count in event_type_counts.items()}

# Apply mapping
data["sdo2_orientation_Event_Types_Attended"] = (
    data["sdo2_orientation_Event_Types_Attended"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.split(",")
    .apply(lambda event_list: [event_type_mapping[etype.strip().lower()] for etype in event_list])
    .apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
)

# Verify transformation
event_types_after = data["sdo2_orientation_Event_Types_Attended"].str.split(", ")
all_event_types_after = [etype for sublist in event_types_after for etype in sublist]
event_type_counts_after = Counter(all_event_types_after)

print("\nEvent types after merging:")
print(event_type_counts_after)

## 4. Validate Results

In [ ]:
# Check all processed columns
for col in cat_cols:
    print(f"\n{col}:")
    print(data[col].value_counts())

print("\nsdo2_orientation_Event_Types_Attended:")
print(data['sdo2_orientation_Event_Types_Attended'].value_counts())

In [ ]:
# For sdo1_previous_Previous_Education_Type adjust 'OVERIG' label to 'Others'
data['sdo1_previous_Previous_Education_Type'] = data['sdo1_previous_Previous_Education_Type'].replace('OVERIG', 'Others')

## 5. Export

In [ ]:
output_path = OUT_DIR / "handled_low_frequency_categories.csv"
data.to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")